# 🛡️ Free Chat Moderation Model Training Pipeline (DistilBERT -> ONNX)
This notebook trains a custom text moderation classifier for **$0** using Google Colab's free GPU, and exports it to **ONNX** format to run directly inside your Node.js backend (`@xenova/transformers`).

In [ ]:
# Step 1: Install required open-source libraries
!pip install -q transformers datasets evaluate accelerate onnx onnxruntime optimum[exporters]

In [ ]:
# Step 2: Upload or define your dataset
import json
from datasets import Dataset

# Either upload your moderation_dataset.json or use the embedded samples:
samples = [
    {"text": "hey! how is your day going?", "label": 0},
    {"text": "just working on my class assignment", "label": 0},
    {"text": "that was an awesome game gg!", "label": 0},
    {"text": "i'm going to kill this workout session today", "label": 0},
    {"text": "kys loser nobody wants you here", "label": 1},
    {"text": "kill yourself you idiot", "label": 1},
    {"text": "m18 looking for girl snapchat me", "label": 1},
    {"text": "add my snap: coolguy_99", "label": 1},
    {"text": "shut up you faggot", "label": 1}
]

dataset = Dataset.from_list(samples)
split_ds = dataset.train_test_split(test_size=0.2, seed=42)
print(f"Training count: {len(split_ds['train'])}, Eval count: {len(split_ds['test'])}")

In [ ]:
# Step 3: Tokenize & Setup DistilBERT
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = split_ds["train"].map(tokenize_fn, batched=True)
tokenized_eval = split_ds["test"].map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "SAFE", 1: "VIOLATION"},
    label2id={"SAFE": 0, "VIOLATION": 1}
)

In [ ]:
# Step 4: Fine-tune the model on free GPU (~2 minutes)
training_args = TrainingArguments(
    output_dir="./distilbert_mod_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()
model.save_pretrained("./saved_pytorch")
tokenizer.save_pretrained("./saved_pytorch")
print("Training complete!")

In [ ]:
# Step 5: Export to ONNX and Download Zip
from optimum.onnxruntime import ORTModelForSequenceClassification
from google.colab import files

onnx_export_path = "./custom_moderator_onnx"
ort_model = ORTModelForSequenceClassification.from_pretrained("./saved_pytorch", export=True)
ort_model.save_pretrained(onnx_export_path)
tokenizer.save_pretrained(onnx_export_path)

!zip -r custom_moderator.zip ./custom_moderator_onnx
files.download("custom_moderator.zip")
print("Downloaded! Unzip and place inside src/data/models/custom_moderator/")